<a href="https://colab.research.google.com/github/Romulus09/TCC/blob/main/TCC_train_test_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
#!pip install datasets mlcroissant
import numpy as np
from datasets import load_dataset,Dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
#import evaluate
import torch


In [24]:
dataset = load_dataset("TimSchopf/medical_abstracts")

# Vamos usar o split "train"
data = dataset["train"]

# =======================================================
# 2. Converter para classificação binária
# =======================================================

TARGET_LABEL = 5

def to_binary(example):
    example["label"] = 1 if example["condition_label"] == TARGET_LABEL else 0
    return example

data = data.map(to_binary)
#for i in range(10):
    #print(data["label"][i])
print(data)
print(data["label"])
print(data[0])
positive_results = Dataset.from_dict({}) # Initialize with dataset features
negative_results = Dataset.from_dict({}) # Initialize with dataset features
#new_data = new_data.add_item(data[0])

for row in (data): # 'row' is already the dictionary for the current example
  if row["label"] == 1:
    positive_results = positive_results.add_item(row)
  else:
    negative_results = negative_results.add_item(row)

Dataset({
    features: ['condition_label', 'medical_abstract', 'label'],
    num_rows: 11550
})
Column([1, 0, 0, 0, 0])
{'condition_label': 5, 'medical_abstract': 'Tissue changes around loose prostheses. A canine model to investigate the effects of an antiinflammatory agent. The aseptically loosened prosthesis provided a means for investigating the in vivo and in vitro activity of the cells associated with the loosening process in seven dogs. The cells were isolated and maintained in culture for sufficient periods of time so that their biologic activity could be studied as well as the effect of different agents added to the cells in vivo or in vitro. The biologic response as determined by interleukin-1 and prostaglandin E2 activity paralleled the roentgenographic appearance of loosening and the technetium images and observations made at the time of revision surgery. The correlation between clinical, roentgenographic, histologic, and biochemical loosening indicates that the canine mode

In [25]:
print(positive_results)
print(negative_results)

Dataset({
    features: ['condition_label', 'medical_abstract', 'label'],
    num_rows: 3844
})
Dataset({
    features: ['condition_label', 'medical_abstract', 'label'],
    num_rows: 7706
})


In [ ]:
# =======================================================
# 2. Tokenizer do RoBERTa
# =======================================================
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

def tokenize(batch):
    return tokenizer(
        batch["medical_abstract"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

encoded = data.map(tokenize, batched=True)

# remover colunas não usadas
#encoded = encoded.remove_columns(["medical_abstract", "condition_label"])

Map:   0%|          | 0/11550 [00:00<?, ? examples/s]

In [ ]:
# =======================================================
# 3. Função de métrica AUC-ROC binária
# =======================================================

def compute_auc_roc(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    auc = roc_auc_score(labels, probs)
    return {"auc_roc": auc}


In [ ]:
from transformers import (
    RobertaTokenizerFast,
    RobertaForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification # Added import
)

# =======================================================
# 4. K-FOLD CROSS VALIDATION (k=5)
# =======================================================
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

# 🔥 limitar dataset
small_dataset = encoded.select(range(min(14000, len(encoded))))

X = np.arange(len(small_dataset))
results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n======== FOLD {fold+1}/{k} ========")

    train_split = small_dataset.select(train_idx)
    eval_split = small_dataset.select(test_idx)

    model = RobertaForSequenceClassification.from_pretrained(
        "roberta-base",
        num_labels=2
    )

    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold}",
        learning_rate=3e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="steps",
        logging_steps=50,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_split,
        eval_dataset=eval_split,
        #tokenizer=tokenizer,
        compute_metrics=compute_auc_roc,
    )

    trainer.train()
    metrics = trainer.evaluate()

    print(f"AUC-ROC do fold {fold+1}: {metrics['eval_auc_roc']:.4f}")
    results.append(metrics["eval_auc_roc"])

# =======================================================
# 6. Resultados finais
# =======================================================
print("\n======= RESULTADOS FINAIS =======")
print("AUC-ROC por fold:", results)
print("Média:", np.mean(results))
print("Desvio padrão:", np.std(results))


======== FOLD 1/5 ========


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Auc Roc
1,0.666200,0.642949,0.521094
2,0.634695,0.645052,0.506580
3,0.644157,0.643934,0.496964


AUC-ROC do fold 1: 0.4970

======== FOLD 2/5 ========


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 